In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt

# Aer is now a separate package (qiskit-aer)
from qiskit_aer import AerSimulator

In [2]:
Q = 8

In [3]:

N = Q*Q # Total Number of vertex in the grid
l = 4/N # Valule for self loop

In [4]:
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)

In [5]:
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")

In [6]:
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)

In [7]:
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')

In [8]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_5: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [9]:
def superposition(circuit, Q):
    num_states = int(2*np.log2(Q))
    for i in range(0,num_states):
        circuit.h(i)

In [10]:
one_step.append(coin_prep, coin)

In [11]:
fig = one_step.decompose().decompose().decompose().decompose()

In [12]:
fig.draw()

global phase: 3π/2
                                                            »
vertex_X_0: ────────────────────────────────────────────────»
                                                            »
vertex_X_1: ────────────────────────────────────────────────»
                                                            »
vertex_X_2: ────────────────────────────────────────────────»
                                                            »
vertex_y_0: ────────────────────────────────────────────────»
                                                            »
vertex_y_1: ────────────────────────────────────────────────»
                                                            »
vertex_y_2: ────────────────────────────────────────────────»
            ┌────────────────┐                         ┌───┐»
    coin_0: ┤ U(π/2,-7π/4,0) ├─────────────────────────┤ X ├»
            ├────────────────┤┌───┐┌──────────────────┐└─┬─┘»
    coin_1: ┤ U(π/2,-7π/4,0) ├┤ X ├┤ U(π/2,π/2,-3π/4) ├──┼──»
            ├────────────────┤└─┬─┘└──────────────────┘  │  »
    coin_2: ┤ U(0.24871,0,0) ├──■────────────────────────■──»
            └────────────────┘                              »
 measure: 6/════════════════════════════════════════════════»
                                                            »
«                                
«vertex_X_0: ────────────────────
«                                
«vertex_X_1: ────────────────────
«                                
«vertex_X_2: ────────────────────
«                                
«vertex_y_0: ────────────────────
«                                
«vertex_y_1: ────────────────────
«                                
«vertex_y_2: ────────────────────
«            ┌──────────────────┐
«    coin_0: ┤ U(π/2,π/2,-3π/4) ├
«            └──────────────────┘
«    coin_1: ────────────────────
«                                
«    coin_2: ────────────────────
«                                
« measure: 6/════════════════════
«

In [13]:
Initialization_count = fig.count_ops()
Initialization_count

OrderedDict([('u', 5), ('cx', 2)])

In [14]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)

In [15]:
one_step.draw()

vertex_X_0: ─────────
                     
vertex_X_1: ─────────
                     
vertex_X_2: ─────────
                     
vertex_y_0: ─────────
                     
vertex_y_1: ─────────
                     
vertex_y_2: ─────────
            ┌───────┐
    coin_0: ┤0      ├
            │       │
    coin_1: ┤1 Coin ├
            │       │
    coin_2: ┤2      ├
            └───────┘
 measure: 6/═════════

In [16]:
one_step.decompose().draw()
fig2 = one_step.decompose()

In [17]:
Coin_Gates = fig2.count_ops()
Coin_Gates

OrderedDict([('u', 26), ('cx', 19), ('rz', 12), ('h', 2)])

In [18]:
def shift(circuit, Q):
    num_states = 3 + int(2*np.log2(Q))
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    circuit.x(num_states-1)
    E = int(np.log2(Q))
    D = E
    for i in range(int(np.log2(Q))):
        x = list(range(0,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(0,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    E = 2*int(np.log2(Q))
    for i in range(int(np.log2(Q))):
        x = list(range(D,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(D,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-1)
    circuit.x(num_states-1)
    circuit.mcx([num_states-1],num_states-3)
    circuit.x(num_states-1)

# 8x8

In [19]:
x = 17 #number of steps

In [20]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [23]:
one_step.decompose().decompose().decompose().count_ops()

OrderedDict([('cx', 9591),
             ('u', 9346),
             ('p', 4785),
             ('t', 480),
             ('rz', 360),
             ('tdg', 360),
             ('h', 300),
             ('crz', 15),
             ('measure', 6),
             ('unitary', 5)])

In [25]:
one_step.decompose().decompose().decompose().depth()

17025

In [27]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# Qiskit gate names that are Clifford for 1q and 2q cases.
# This covers the common standard gates from Qiskit's gate set.
CLIFFORD_GATES = {
    # 1-qubit Clifford
    "id", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    # 2-qubit Clifford
    "cx", "cy", "cz", "swap",
    # you may also want to treat barriers/measure/reset as neutral
    "barrier", "measure", "reset"
}

def is_non_clifford_instruction(inst):
    """
    Return True if a Qiskit instruction is non-Clifford.
    """
    return inst.name not in CLIFFORD_GATES

def non_clifford_depth(qc: QuantumCircuit) -> int:
    """
    Compute the depth contributed by non-Clifford gates only.

    A DAG layer is counted if it contains at least one non-Clifford operation.
    """
    dag = circuit_to_dag(qc)
    depth = 0

    for layer in dag.layers():
        layer_dag = layer["graph"]
        has_non_clifford = any(
            is_non_clifford_instruction(node.op)
            for node in layer_dag.op_nodes()
            if node.op.name not in {"barrier", "measure", "reset"}
        )
        if has_non_clifford:
            depth += 1

    return depth

# Example
      # non-Clifford

#print(one_step)
print("Total circuit depth:", one_step.decompose().decompose().decompose().depth())
print("Non-Clifford depth:", non_clifford_depth(one_step.decompose().decompose().decompose()))

Total circuit depth: 17025
Non-Clifford depth: 11062


for shift

In [26]:
Q = 8
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()

OrderedDict([('cx', 129),
             ('h', 120),
             ('p', 60),
             ('cp', 36),
             ('t', 32),
             ('tdg', 32),
             ('u', 10),
             ('mcphase', 4)])

# 16x16

In [28]:
Q = 16
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [29]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_5: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_6: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_7: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [30]:
x = 33
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [32]:
one_step.decompose().decompose().decompose().count_ops()

OrderedDict([('cx', 41375),
             ('u', 34396),
             ('p', 22103),
             ('t', 1984),
             ('h', 1736),
             ('tdg', 1612),
             ('rz', 1240),
             ('crz', 31),
             ('measure', 8),
             ('unitary', 5)])

In [33]:
one_step.decompose().decompose().decompose().depth()

70001

In [34]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# Qiskit gate names that are Clifford for 1q and 2q cases.
# This covers the common standard gates from Qiskit's gate set.
CLIFFORD_GATES = {
    # 1-qubit Clifford
    "id", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    # 2-qubit Clifford
    "cx", "cy", "cz", "swap",
    # you may also want to treat barriers/measure/reset as neutral
    "barrier", "measure", "reset"
}

def is_non_clifford_instruction(inst):
    """
    Return True if a Qiskit instruction is non-Clifford.
    """
    return inst.name not in CLIFFORD_GATES

def non_clifford_depth(qc: QuantumCircuit) -> int:
    """
    Compute the depth contributed by non-Clifford gates only.

    A DAG layer is counted if it contains at least one non-Clifford operation.
    """
    dag = circuit_to_dag(qc)
    depth = 0

    for layer in dag.layers():
        layer_dag = layer["graph"]
        has_non_clifford = any(
            is_non_clifford_instruction(node.op)
            for node in layer_dag.op_nodes()
            if node.op.name not in {"barrier", "measure", "reset"}
        )
        if has_non_clifford:
            depth += 1

    return depth

# Example
      # non-Clifford

#print(one_step)
print("Total circuit depth:", one_step.decompose().decompose().decompose().depth())
print("Non-Clifford depth:", non_clifford_depth(one_step.decompose().decompose().decompose()))

Total circuit depth: 70001
Non-Clifford depth: 44167


for shift

In [34]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()

OrderedDict([('cx', 129),
             ('h', 128),
             ('p', 60),
             ('cp', 36),
             ('t', 32),
             ('tdg', 32),
             ('u', 10),
             ('mcphase', 8)])

# 32x32

In [35]:
Q = 32
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [36]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
#phase_circuit.draw()

In [37]:
x = 75
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [38]:
one_step.decompose().decompose().decompose().depth()

303336

In [39]:
one_step.decompose().decompose().decompose().count_ops()

OrderedDict([('cx', 177995),
             ('u', 133494),
             ('p', 92929),
             ('t', 14600),
             ('tdg', 12556),
             ('h', 11680),
             ('rz', 4088),
             ('crz', 73),
             ('measure', 10),
             ('unitary', 5)])

In [40]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# Qiskit gate names that are Clifford for 1q and 2q cases.
# This covers the common standard gates from Qiskit's gate set.
CLIFFORD_GATES = {
    # 1-qubit Clifford
    "id", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    # 2-qubit Clifford
    "cx", "cy", "cz", "swap",
    # you may also want to treat barriers/measure/reset as neutral
    "barrier", "measure", "reset"
}

def is_non_clifford_instruction(inst):
    """
    Return True if a Qiskit instruction is non-Clifford.
    """
    return inst.name not in CLIFFORD_GATES

def non_clifford_depth(qc: QuantumCircuit) -> int:
    """
    Compute the depth contributed by non-Clifford gates only.

    A DAG layer is counted if it contains at least one non-Clifford operation.
    """
    dag = circuit_to_dag(qc)
    depth = 0

    for layer in dag.layers():
        layer_dag = layer["graph"]
        has_non_clifford = any(
            is_non_clifford_instruction(node.op)
            for node in layer_dag.op_nodes()
            if node.op.name not in {"barrier", "measure", "reset"}
        )
        if has_non_clifford:
            depth += 1

    return depth

# Example
      # non-Clifford

#print(one_step)
print("Total circuit depth:", one_step.decompose().decompose().decompose().depth())
print("Non-Clifford depth:", non_clifford_depth(one_step.decompose().decompose().decompose()))

Total circuit depth: 303336
Non-Clifford depth: 186417


for shift


In [75]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()

OrderedDict([('h', 136),
             ('cx', 129),
             ('p', 60),
             ('cp', 36),
             ('t', 32),
             ('tdg', 32),
             ('mcphase', 12),
             ('u', 10)])

# 64x64

In [41]:
Q = 64
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [42]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
#phase_circuit.draw()

In [43]:
x = 165
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [44]:
one_step.decompose().decompose().decompose().depth()

1136687

In [45]:
one_step.decompose().decompose().decompose().count_ops()

OrderedDict([('cx', 655607),
             ('u', 446672),
             ('p', 379627),
             ('t', 59984),
             ('tdg', 52812),
             ('h', 48248),
             ('rz', 11736),
             ('crz', 163),
             ('measure', 12),
             ('unitary', 5)])

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# Qiskit gate names that are Clifford for 1q and 2q cases.
# This covers the common standard gates from Qiskit's gate set.
CLIFFORD_GATES = {
    # 1-qubit Clifford
    "id", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    # 2-qubit Clifford
    "cx", "cy", "cz", "swap",
    # you may also want to treat barriers/measure/reset as neutral
    "barrier", "measure", "reset"
}

def is_non_clifford_instruction(inst):
    """
    Return True if a Qiskit instruction is non-Clifford.
    """
    return inst.name not in CLIFFORD_GATES

def non_clifford_depth(qc: QuantumCircuit) -> int:
    """
    Compute the depth contributed by non-Clifford gates only.

    A DAG layer is counted if it contains at least one non-Clifford operation.
    """
    dag = circuit_to_dag(qc)
    depth = 0

    for layer in dag.layers():
        layer_dag = layer["graph"]
        has_non_clifford = any(
            is_non_clifford_instruction(node.op)
            for node in layer_dag.op_nodes()
            if node.op.name not in {"barrier", "measure", "reset"}
        )
        if has_non_clifford:
            depth += 1

    return depth

# Example
      # non-Clifford

#print(one_step)
print("Total circuit depth:", one_step.decompose().decompose().decompose().depth())
print("Non-Clifford depth:", non_clifford_depth(one_step.decompose().decompose().decompose()))

Total circuit depth: 1136687


In [85]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
#one_step.draw()

In [72]:
one_step.decompose().count_ops()

OrderedDict([('h', 144),
             ('cx', 129),
             ('p', 60),
             ('cp', 36),
             ('t', 32),
             ('tdg', 32),
             ('mcphase', 16),
             ('u', 10)])